# Documentary GPU — 音楽制作（理論based劇伴）

設計: `docs/music_system.md` / `docs/music_schema.md`

**思想**: 理論で書いたMIDIスコアが背骨、AIは質感のみ。キューベース。

工程: cue(JSON) → MIDI(music21) → 楽器音化(FluidSynth) → ミックス(pedalboard) → -16 LUFS(pyloudnorm)

※MP2はGPU不要（CPUランタイムで可）。AIテクスチャ(MP5)時のみGPU。

In [ ]:
# ═══ セル1: コード取得 ═══
import os, subprocess
WORK = '/content/documentary_gpu'
REPO = 'https://github.com/task2000jp/documentary_gpu.git'
if os.path.isdir(WORK):
    subprocess.run(['git', '-C', WORK, 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', REPO, WORK], check=True)
print('WORK =', WORK)

In [ ]:
# ═══ セル2: 依存（音楽スタック）═══
%cd $WORK
# FluidSynth本体 + 無料GM SoundFont（orchestral含む）
!apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
!pip install -q music21 pedalboard pyloudnorm soundfile numpy
print('音楽スタック導入完了')

In [ ]:
# ═══ セル3: 作曲＋レンダリング（cue → MIDI → WAV を1コマンドで）═══
%cd $WORK
CUE = 'proto_knopfler'   # cues/<名>.json の <名>。管弦の勝利主題は 'ch6'
# style は cue の "style" から自動判定（fingerstyle=Knopfler風 / orchestral=管弦）。
# 上書きするなら末尾に --style orchestral などを足す。
import json
cid = json.load(open(f'cues/{CUE}.json'))['id']
WAV = f'build/music/{cid}.wav'
!python scripts/pipeline.py music $CUE

In [ ]:
# ═══ セル4: 試聴 ═══
from IPython.display import Audio
Audio(WAV)

In [ ]:
# ═══ セル5: Driveへ保存 ═══
from google.colab import drive; drive.mount('/content/drive')
import shutil, glob, os
OUT = '/content/drive/MyDrive/documentary_gpu_output/music'
os.makedirs(OUT, exist_ok=True)
for f in glob.glob('build/music/*.wav'):
    shutil.copy(f, OUT); print('保存:', os.path.basename(f))
print('→', OUT)